In [ ]:
# Imports
!apt-get install -y timidity
!pip install midi2audio
!pip install -q pretty_midi
!pip install -q pypianoroll

import pretty_midi
from midi2audio import FluidSynth
from IPython.display import Audio

import pypianoroll
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

import os
from tqdm.notebook import tqdm
import cv2 as cv
from PIL import Image

from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import torch.nn as nn
import torch
import torch.optim as optim
import torch.autograd as autograd

In [ ]:
# Define Sizes

batch_size = 32
num_workers = 4
inp_dim = 128

path = "/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz"
multitrack = pypianoroll.load(path)
track_names = ["Drums", "Bass", "Guitar", "Strings", "Piano"]

In [ ]:
path = "/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz"

multitrack = pypianoroll.load(path)

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True, sharey=True)

track_names = ["Drums", "Bass", "Guitar", "Strings", "Piano"]

for i in range(5):
    roll = multitrack.tracks[i].pianoroll.astype(np.int32)
    axes[i].imshow(roll.T, aspect="auto", origin="lower", cmap="gray_r")
    axes[i].set_ylabel(track_names[i])
    if i == 0:
        axes[i].set_title("Multitrack Pianoroll")

In [ ]:
class PRDataset(Dataset):
    def __init__(self, root_dir, tracks = 5, t_len = 96*4,pitch_size=128, transform = None):
        self.root_dir = root_dir
        self.transform = transform
        self.t_len = t_len
        self.samples = []
        self.pitch_size = pitch_size
        self.tracks= tracks
        for root, _, files in tqdm(os.walk(self.root_dir)):
            for file in files:
                if file.lower().endswith('.npz'):
                    multitrack = pypianoroll.load(os.path.join(root,file))
                    T = multitrack.tracks[0].pianoroll.shape[0]
                    n_samples = T // self.t_len 
                    for i in range(n_samples):
                       self.samples.append((os.path.join(root,file), i *  self.t_len))
        
    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, start = self.samples[idx]
        multitrack = pypianoroll.load(path)
        rolls = []
        for track in multitrack.tracks:
            pr = track.pianoroll[start:start+self.t_len]
            if pr.shape[0] < self.t_len:
                pad_width = ((0, self.t_len - pr.shape[0]), (0, 0))
                pr = np.pad(pr, pad_width, mode="constant")
            if pr.shape[1] < self.pitch_size:
                pad_width = ((0, 0), (0, self.pitch_size - pr.shape[1]))
                pr = np.pad(pr, pad_width, mode="constant")
            elif pr.shape[1] > self.pitch_size:
                pr = pr[:, :self.pitch_size]
                
            rolls.append(pr)
        while len(rolls) < self.tracks:
            rolls.append(np.zeros((self.t_len, self.pitch_size)))
        
        rolls = np.stack(rolls, axis=0)  
        rolls = torch.tensor(rolls, dtype=torch.float32)  / 127.0
        if self.transform:
            return self.transform(rolls)
        return rolls

In [ ]:
# # Nope

# class PRDataset(Dataset):
#     def __init__(self, root_dir, tracks = 5, t_len = 96*4, transform = None):
#         self.root_dir = root_dir
#         self.transform = transform
#         self.t_len = t_len
#         self.samples = []
#         npz_files = []
        
#         for root, _, files in os.walk(self.root_dir):
#             for file in files:
#                 if file.lower().endswith('.npz'):
#                     npz_files.append(os.path.join(root, file))
                    
#         print(f"Found {len(npz_files)} files. Indexing samples...")
        
#         for file_path in tqdm(npz_files, desc="Indexing samples"):
#             multitrack = pypianoroll.load(file_path)
#             T = multitrack.tracks[0].pianoroll.shape[0]
#             n_samples = T // self.t_len   
#             if n_samples > 0:
#                 for i in range(n_samples):
#                     self.samples.append((file_path, i * self.t_len))
    
#     def __len__(self):
#         return len(self.samples)
        
#     def __getitem__(self, idx):
#         path, start = self.samples[idx]
#         multitrack = pypianoroll.load(path)
#         rolls = []
#         for track in multitrack.tracks:
#             pr = track.pianoroll[start:start+self.t_len] 
#             rolls.append(pr)

        
#         rolls = np.stack(rolls, axis=0)  
#         rolls = torch.tensor(rolls, dtype=torch.float32)  / 127.0
#         if self.transform:
#             return self.transform(rolls)
#         return rolls

In [ ]:
transform = transforms.Compose([
    transforms.Normalize((0.5,)*5, (0.5,)*5)     
])

root_dir = "/kaggle/input/lpd-5-cleansed/"

dataset = PRDataset(root_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

In [ ]:
# def play_sample(path):
#     mt = pypianoroll.load(path)
#     pm = mt.to_pretty_midi()
#     pm.write(f"/kaggle/working/temp.mid")
#     !timidity /kaggle/working/temp.mid -Ow -o /kaggle/working/temp.wav -q0
#     display(Audio("/kaggle/working/temp.wav"))

# play_sample("/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz")
# play_sample("/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/W/E/D/TRWEDUK12903D0936E/c2e28f932dcc7eb5487587fa6be72d92.npz")

In [ ]:
# # Load multitrack
# mt = pypianoroll.load("/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz")

# # Convert to PrettyMIDI
# pm = mt.to_pretty_midi()

# # Save to a MIDI file
# pm.write("/kaggle/working/temp.mid")

# # Render to WAV using timidity
# !timidity /kaggle/working/temp.mid -Ow -o /kaggle/working/temp.wav

# Audio("/kaggle/working/temp.wav")

In [ ]:
# For the critic
def wasserstein_critic_loss(real_output, fake_output):
    return torch.mean(fake_output) - torch.mean(real_output)

# For the generator
def wasserstein_generator_loss(fake_output):
    return -torch.mean(fake_output)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Gradient Penalty

def gradient_penalty(critic, real_data, fake_data, device=device, lambda_gp=10):
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    alpha = alpha.expand_as(real_data)

    if real_data.shape != fake_data.shape:
        fake_data = F.interpolate(fake_data, size=real_data.shape[2:], mode='bilinear', align_corners=False)
    
    interpolate = alpha * real_data + (1 - alpha) * fake_data
    interpolate = interpolate.to(device).requires_grad_(True)

    interpolated_score = critic(interpolate)

    gradients = autograd.grad(
        outputs=interpolated_score,
        inputs=interpolate,
        grad_outputs=torch.ones_like(interpolated_score),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)

    gradient_penalty = lambda_gp * ((gradient_norm - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
# gp = compute_gradient_penalty(critic, real_data, fake_data, device=device)

# generator_loss = wasserstein_generator_loss(fake_data) + gp
# critic_loss = wasserstein_critic_loss(real_data, fake_data) + gp

In [ ]:
class TrackGenerator(nn.Module):
    def __init__(self, input_dim, out_dim, hidden_dim, pitch_range, time_res):
        super().__init__()

        self.pitch_range = pitch_range
        self.time_res = time_res

        self.project = nn.Sequential(
            
            nn.Linear(input_dim, 256 * 6 * 8),
            nn.ReLU()
        )
        
        self.deconv = nn.Sequential(
            
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # (128, 12, 16)
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # (64, 24, 32)
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),     # (1, 48, 64)
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.project(x)                   # (batch, 256*6*8)
        x = x.view(batch_size, 256, 6, 8)     # (batch, 256, 6, 8)
        x = self.deconv(x)                    # (batch, 1, pitch_range, time_res)
        x = F.interpolate(x, size=(self.pitch_range, self.time_res), mode='bilinear', align_corners=False)
        return x.squeeze(1)                   # (batch, pitch_range, time_res)

In [ ]:
class BarGenerator(nn.Module) :
    def __init__(self, input_dim, out_dim, hidden_dim, pitch_range, time_res):
        super().__init__()

        self.pitch_range = pitch_range
        self.time_res = time_res

        self.model = nn.Sequential(
            
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, out_dim),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.model(x)   # [B, out_dim]

In [ ]:
class HybridMuseGenerator(nn.Module):
    def __init__(self, latent_dim=128, z_t_dim=32, z_i_dim=32, z_it_dim=32, time_steps=4, num_tracks=5, pitch_range=128, time_res=96, hidden_dim=256):
        super().__init__()

        self.latent_dim = latent_dim
        self.z_t_dim = z_t_dim
        self.z_i_dim = z_i_dim
        self.z_it_dim = z_it_dim
        self.time_steps = time_steps
        self.num_tracks = num_tracks
        self.pitch_range = pitch_range
        self.time_res = time_res

        self.out_dim = pitch_range * time_res

        self.bar_generator = BarGenerator(
                input_dim = z_t_dim,
                out_dim = hidden_dim,
                hidden_dim = hidden_dim,
                pitch_range = self.pitch_range,
                time_res = self.time_res
            )

        self.track_generators = nn.ModuleList([
            TrackGenerator(
                input_dim = z_i_dim + z_it_dim + hidden_dim,
                out_dim = self.out_dim,
                hidden_dim = hidden_dim,
                pitch_range = self.pitch_range,
                time_res = self.time_res
            ) for _ in range(num_tracks)
        ])

    def forward(self, z_t, z_i_list, z_it_list):
        batch_size = z_t.size(0)
        
        temporal_context = []

        for t in range(self.time_steps) :
            z_t_step = z_t[:, t, :]
            context_t = self.bar_generator(z_t_step)
            temporal_context.append(context_t)
        
        outputs = []

        for track_idx in range(self.num_tracks):
            z_i = z_i_list[track_idx]
            track_outputs = []
            
            for t in range(self.time_steps):
                z_it = z_it_list[track_idx][t]
                
                combine_ip = torch.cat([z_i, z_it, temporal_context[t]], dim=1)
                
                pianoroll = self.track_generators[track_idx](combine_ip)

                track_outputs.append(pianoroll)
            track_outputs = torch.stack(track_outputs, dim=1)
            outputs.append(track_outputs)

        outputs = torch.stack(outputs, dim=1)
        batch_size, num_tracks, time_steps, pitch_range, time_res = outputs.shape
        
        # output shape: [B, N, T, P, R]
        outputs = outputs.permute(0, 1, 3, 2, 4)  # [B, N, P, T, R]
        outputs = outputs.reshape(batch_size, self.num_tracks, self.pitch_range, self.time_steps * self.time_res)

        return outputs

In [ ]:
# Discriminator

class Discriminator(nn.Module):
    def __init__(self, inc=5):
        super(Discriminator, self).__init__()
        
        self.model = nn.Sequential(
            nn.Conv2d(inc, 64, kernel_size=4, stride=2, padding=1), 
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True), 
            
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        out = self.model(x)      
        return out.view(out.size(0), -1).mean(dim=1, keepdim=True)

In [ ]:
# Initialize models

generator = HybridMuseGenerator().to(device)
discriminator = Discriminator().to(device)

In [ ]:
opt_G = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.9))
opt_D = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.9))

In [ ]:
# Training Loop
num_epochs = 1

z_t_dim = 32
z_i_dim = 32 
z_it_dim = 32
time_steps = 4 
num_tracks = 5 
pitch_range = 128
time_res = 96
hidden_dim = 256
batch_size = 64
lambda_l1 = 10
GLosses = []
Dlosses = []

n_critic = 1 #No. of epochs of Discriminator per Generator epoch
for epoch in range(num_epochs):
    for i, real in enumerate(tqdm(dataloader, leave=False)):
        real = real.to(device)
        if real.shape[-1] != time_steps * time_res:
            real = F.interpolate(real, size=(pitch_range, time_steps * time_res), mode='bilinear', align_corners=False)
        
        for _ in range(n_critic):
            z_t = torch.randn(batch_size, time_steps, z_t_dim, device=device)
            z_i_list = [torch.randn(batch_size, z_i_dim, device=device) for _ in range(num_tracks)]
            z_it_list = [
                [torch.randn(batch_size, z_it_dim, device=device) for _ in range(time_steps)]
                for _ in range(num_tracks)
            ]
            fake = generator(z_t, z_i_list, z_it_list)

            if real.shape[-1] != fake.shape[-1]:
                real = F.interpolate(real, size=fake.shape[2:], mode='bilinear', align_corners=False)
            
            real_score = discriminator(real)
            fake_score = discriminator(fake.detach())

            gp = gradient_penalty(discriminator, real, fake.detach(), device)

            d_loss = wasserstein_critic_loss(real_score, fake_score) + gp

            opt_D.zero_grad()
            d_loss.backward()
            opt_D.step()
            
        z_t = torch.randn(batch_size, time_steps, z_t_dim, device=device)
        z_i_list = [torch.randn(batch_size, z_i_dim, device=device) for _ in range(num_tracks)]
        z_it_list = [
            [torch.randn(batch_size, z_it_dim, device=device) for _ in range(time_steps)]
                for _ in range(num_tracks)
        ]
        fake_score = discriminator(generator(z_t, z_i_list, z_it_list))
        g_loss = wasserstein_generator_loss(fake_score)

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()
        
    if (epoch+1) % 1 == 0:
        z_t = torch.randn(batch_size, time_steps, z_t_dim, device=device)
        z_i_list = [torch.randn(batch_size, z_i_dim, device=device) for _ in range(num_tracks)]
        z_it_list = [
                [torch.randn(batch_size, z_it_dim, device=device) for _ in range(time_steps)]
                for _ in range(num_tracks)
            ]
        fake = generator(z_t, z_i_list, z_it_list).detach().cpu().numpy()[0]   # shape: (tracks, time, pitch)
        programs = [0, 32, 24, 48, 0]  # Piano, Bass, Guitar, Strings, Drums
        is_drum = [False, False, False, False, True]
        mt = pypianoroll.Multitrack(tracks=[
            pypianoroll.Track(pianoroll=fake_sample[inst], program=programs[inst], is_drum=is_drum[inst])
            for inst in range(fake_sample.shape[0])
        ])
        
        fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True, sharey=True)

        track_names = ["Drums", "Bass", "Guitar", "Strings", "Piano"]

        for i in range(5):
            roll = mt.tracks[i].pianoroll.astype(np.int32)
            axes[i].imshow(roll.T, aspect="auto", origin="lower", cmap="gray_r")
            axes[i].set_ylabel(track_names[i])
            if i == 0:
                axes[i].set_title("Multitrack Pianoroll")
        path = f"/kaggle/working/sample_epoch{epoch}.npz"
        mt.save(path)
    
        print(f"Saved sample: {path}")

        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'loss_G': GLosses,
            'loss_D': Dlosses,
            }, f"/kaggle/working/gan_checkpoint_epoch_{epoch+1}.pth")

In [ ]:
z_t = torch.randn(batch_size, time_steps, z_t_dim, device=device)
z_i_list = [torch.randn(batch_size, z_i_dim, device=device) for _ in range(num_tracks)]
z_it_list = [
    [torch.randn(batch_size, z_it_dim, device=device) for _ in range(time_steps)]
    for _ in range(num_tracks)
]
fake = generator(z_t, z_i_list, z_it_list)
sample = fake[0].detach().cpu().numpy()   # shape: (num_tracks, time, pitch)

num_tracks = sample.shape[0]
fig, axes = plt.subplots(num_tracks, 1, figsize=(12, 2*num_tracks), sharex=True)

for i in range(num_tracks):
    axes[i].imshow(sample[i].T, aspect="auto", origin="lower", cmap="gray")
    axes[i].set_ylabel(f"Track {i+1}", fontsize=8)
    if i == num_tracks - 1:
        axes[i].set_xlabel("Time")

plt.tight_layout()
plt.show()

In [ ]:
# def rolls_to_midi(rolls, filepath="sample.mid", tempo=120, resolution=24):
#     """
#     rolls: numpy array, shape (num_tracks, time, pitch), values in [0,1]
#     """
#     tracks = []
#     for i, pr in enumerate(rolls):
#         pr = np.clip(pr, 0, 1)
#         pr = (pr * 127).astype(np.uint8)   # scale back to MIDI velocity range
#         track = pypianoroll.StandardTrack(
#             pianoroll=pr,
#             program=0,          # 0 = acoustic grand piano
#             is_drum=False,
#             name=f"Track {i}"
#         )
#         tracks.append(track)

#     multitrack = pypianoroll.Multitrack(
#         tracks=tracks,
#         tempo=np.array([tempo]),   # <-- must be numpy array
#         resolution=resolution
#     )
#     multitrack.write(filepath)
#     print(f"Saved MIDI to {filepath}")


# # Example: take first fake sample
# sample = fake[0].detach().cpu().numpy()
# rolls_to_midi(sample, "generated.mid")


In [ ]:
def rolls_to_midi(rolls, filepath="sample.mid", tempo=120, resolution=24):
    """
    rolls: numpy array, shape (num_tracks, time, pitch), values in [0,1]
    """
    tracks = []
    for i, pr in enumerate(rolls):
        pr = np.clip(pr, 0, 1)
        pr = (pr * 127).astype(np.uint8)   # scale to MIDI velocity range [0-127]
        
        print(f"Track {i} min/max: {pr.min()}/{pr.max()}")  # debug info
        
        track = pypianoroll.StandardTrack(
            pianoroll=pr,
            program=0,
            is_drum=False,
            name=f"Track {i}"
        )
        tracks.append(track)

    multitrack = pypianoroll.Multitrack(
        tracks=tracks,
        tempo=np.array([tempo]),
        resolution=resolution
    )
    multitrack.write(filepath)
    print(f"Saved MIDI to {filepath}")

In [ ]:
!timidity /kaggle/working/generated.mid -Ow -o /kaggle/working/temp.wav -q0
Audio("/kaggle/working/temp.wav")

In [ ]:
plt.plot(Dlosses)
plt.plot(GLosses)
plt.show()